# Chapter 5 — The Three-Stage Tree-and-Reasoning Architecture

> Architect · Reasoner · Answerer + Verifier · Production-grade
> v2.0 / 2026 · NOWAVE

## 튜토리얼 구성

| § | 튜토리얼 | 핵심 |
|---|---|---|
| 5-1 | 3 Typed Schemas | `ArchitectDecision`·`TraversalDecision`·`GroundedAnswer` |
| 5-2 | Stage 1 — Architect (raw/summary 분리) | parallel 호출, JSON 트리 + raw 파일 |
| 5-3 | Stage 2 — Reasoner (budget + trace) | LangGraph 추론 루프, JSONL trace |
| 5-4 | Stage 3 — Answerer + Verifier (fail closed) | 5종 결정적 검사, refusal 패턴 |

## 0. 환경 준비

```bash
pip install -q openai langgraph pydantic python-dotenv pymupdf pymupdf4llm
```

본 노트북 비용은 약 $3~$10 (10K 분량 문서 1개, 5개 쿼리).

In [24]:
# ─────────────────────────────────────────────────────
# 환경 변수 + 모델 분리 설정 (Architect=mini, Reasoner/Answerer=4.1)
# ─────────────────────────────────────────────────────
import os, json, time, re
from pathlib import Path
from typing import Optional, Literal, Annotated, TypedDict
import operator

from dotenv import load_dotenv
from openai import OpenAI
from pydantic import BaseModel, Field

load_dotenv()
#%os.environ.setdefault("OPENAI_API_KEY", "sk-...")

client = OpenAI()

# 모델 분리 — Architect=mini, Reasoner/Answerer=4.1 (데모는 모두 mini로 단순화)
MODEL_ARCHITECT = "gpt-5.4-mini"
MODEL_REASONER  = "gpt-5.4-mini"       # 프로덕션은 gpt-4.1 권장
MODEL_ANSWERER  = "gpt-5.4-mini"       # 동일

WORK = Path("./work"); WORK.mkdir(exist_ok=True)
RES  = Path("./work/results"); RES.mkdir(parents=True, exist_ok=True)
RAW_DIR = WORK / "raw"; RAW_DIR.mkdir(exist_ok=True)
TRACE_PATH = WORK / "traversal.jsonl"

print("환경 준비 완료")

환경 준비 완료


---
## §5-1 Three Typed Schemas

Pydantic의 `BaseModel`로 3-Stage 각 단계의 LLM 출력을 schema로 강제한다. `response_format=Model`로 OpenAI에 전달하면 schema 위반이 0%가 된다.

이 schema 정의가 Chapter 4의 dataclass와 본 챕터의 가장 큰 차이 — Chapter 4는 자유로운 LLM 응답을 사후 파싱했다면, 본 챕터는 schema를 사전 강제한다.

### 5-1-1. Stage 1 schema — ArchitectDecision · TreeNode

Architect 단계의 입력은 raw section text, 출력은 routing 메타데이터(title, summary, has_table, suggested_node_id)다. 4개 필드만 허용함으로써 LLM의 prose drift를 차단한다.

In [25]:
# ─────────────────────────────────────────────────────
# §5-1-1: Stage 1 schema 정의
# ─────────────────────────────────────────────────────
class ArchitectDecision(BaseModel):
    """LLM이 한 섹션에 대해 반환할 수 있는 것만 정의 (prose drift 방지)"""
    title:             str = Field(min_length=1, max_length=200)
    summary:           str = Field(min_length=1, max_length=600)
    has_table:         bool
    suggested_node_id: str = Field(pattern=r"^[a-z0-9_]+$")

class TreeNode(BaseModel):
    """저장 모델 — raw text는 별도 파일 (load_raw_section으로 읽음)"""
    node_id:     str = Field(pattern=r"^[a-z0-9_]+(\.[a-z0-9_]+)*$")
    title:       str
    summary:     str
    page_range:  tuple[int, int]
    parent_path: list[str] = Field(default_factory=list)
    children:    list[str] = Field(default_factory=list)
    has_table:   bool = False
    source_refs: list[str] = Field(default_factory=list)

print("Stage 1 schemas:", ArchitectDecision.__name__, "→", TreeNode.__name__)

Stage 1 schemas: ArchitectDecision → TreeNode


### 5-1-2. Stage 2 schema — CandidateView · TraversalDecision

Reasoner는 raw text를 보지 못한다 — title/summary/page_range/has_table만 노출하여 컨텍스트 폭주를 막고 라우팅 의사결정을 강제한다.

In [26]:
# ─────────────────────────────────────────────────────
# §5-1-2: Stage 2 schema 정의
# ─────────────────────────────────────────────────────
class CandidateView(BaseModel):
    """Reasoner가 볼 수 있는 정보 — raw text 제외 (의도적 차단)"""
    node_id:     str
    title:       str
    summary:     str
    page_range:  tuple[int, int]
    has_table:   bool
    parent_path: list[str]

class TraversalDecision(BaseModel):
    """Reasoner의 출력 — 다음 단계 결정"""
    select_for_evidence: list[str] = Field(default_factory=list)
    expand_next:         list[str] = Field(default_factory=list)
    reason:              str = Field(max_length=400)
    has_enough_evidence: bool
    evidence_gap:        str = Field(default="", max_length=400)

print("Stage 2 schemas:", CandidateView.__name__, "→", TraversalDecision.__name__)

Stage 2 schemas: CandidateView → TraversalDecision


### 5-1-3. Stage 3 schema — Citation · GroundedAnswer · VerifyResult

Answerer의 출력은 인용을 강제하는 GroundedAnswer다. Citation은 node_id + page + quote의 3-tuple로, Verifier가 substring 검증할 수 있는 형식이다.

In [27]:
# ─────────────────────────────────────────────────────
# §5-1-3: Stage 3 schema 정의
# ─────────────────────────────────────────────────────
class Citation(BaseModel):
    """인용 1개 — Verifier가 substring 검증하는 형식"""
    node_id: str
    page:    int
    quote:   str = Field(min_length=1, max_length=400)

class GroundedAnswer(BaseModel):
    """Answerer의 출력 — 인용·신뢰도·missing 필수"""
    answer:           str = Field(min_length=1, max_length=2000)
    calculation:      str = Field(default="", max_length=600)
    citations:        list[Citation]
    confidence:       Literal["high","medium","low"] = "medium"
    missing_evidence: list[str] = Field(default_factory=list)

class VerifyResult(BaseModel):
    """Verifier의 출력 — ok 플래그 + 실패 사유 리스트"""
    ok:       bool
    failures: list[str] = Field(default_factory=list)

print("3개 단계 6개 schema 정의 완료")
print(f"  Stage 1: {ArchitectDecision.__name__} → {TreeNode.__name__}")
print(f"  Stage 2: {CandidateView.__name__} → {TraversalDecision.__name__}")
print(f"  Stage 3: {GroundedAnswer.__name__} + {VerifyResult.__name__}")

3개 단계 6개 schema 정의 완료
  Stage 1: ArchitectDecision → TreeNode
  Stage 2: CandidateView → TraversalDecision
  Stage 3: GroundedAnswer + VerifyResult


---
## §5-2 Stage 1 — Architect (raw/summary 분리)

각 섹션마다 LLM을 1회 호출하여 `ArchitectDecision`을 받고, raw text는 별도 파일로 저장한다. 답변 단계에서 summary가 아닌 raw text를 인용하기 위함이다.

**3원칙**:
1. ArchitectDecision은 4개 필드만 반환 (prose drift 방지)
2. raw text는 별도 파일에 저장 (답변 시 summary가 아닌 원본 참조)
3. summary는 라우팅 메타데이터 — 숫자 paraphrase 금지

### 5-2-1. 데모 입력 — 합성 SEC 10-K 섹션 5개

실전에서는 Chapter 1의 Docling 결과를 사용하지만, 데모 재현성을 위해 핵심 섹션 5개를 직접 정의한다.

In [28]:
# ─────────────────────────────────────────────────────
# §5-2-1: 데모 입력 — 합성 SEC 10-K 섹션 5개
# ─────────────────────────────────────────────────────
SECTIONS = [
    {
        "heading": "Item 7. Management Discussion and Analysis",
        "page_range": [25, 45],
        "parent_path": [],
        "raw_text": "This Management's Discussion section reviews FY2024 results. "
                    "We discuss revenue, operating income, and segment performance. "
                    "See Results of Operations for quarterly detail.",
    },
    {
        "heading": "Results of Operations",
        "page_range": [30, 42],
        "parent_path": ["mdna"],
        "raw_text": "Our results of operations include quarterly performance. "
                    "Total FY2024 operating income was $52,789M. See quarterly table below.",
    },
    {
        "heading": "Quarterly Operating Income",
        "page_range": [41, 43],
        "parent_path": ["mdna", "results_of_operations"],
        "raw_text": "Quarter | OpInc | OpInc ex-restructuring\n"
                    "Q3 2023 | $13,453M | $14,201M\n"
                    "Q3 2024 | $14,667M | $15,890M\n"
                    "Excluding restructuring charges, Q3 2024 OpInc rose 11.9% YoY.",
    },
    {
        "heading": "Restructuring Charges",
        "page_range": [44, 44],
        "parent_path": ["mdna", "results_of_operations"],
        "raw_text": "Restructuring charges totaled $748M in Q3 2023 and $1,223M in Q3 2024, "
                    "primarily related to workforce optimization.",
    },
    {
        "heading": "Item 1A. Risk Factors",
        "page_range": [15, 24],
        "parent_path": [],
        "raw_text": "Our business faces supply chain, regulatory, and competitive risks. "
                    "Material adverse events could impact future results.",
    },
]
print(f"데모 섹션 {len(SECTIONS)}개 준비")

데모 섹션 5개 준비


### 5-2-2. Architect 함수 — 섹션 1개 → TreeNode 1개

각 섹션마다 LLM 1회 호출. `response_format=ArchitectDecision`이 schema 위반을 컴파일 시간에 차단한다.

In [29]:
# ─────────────────────────────────────────────────────
# §5-2-2: Architect 함수 + 실행
# ─────────────────────────────────────────────────────
_ARCHITECT_PROMPT = """You are indexing a structured document.
Given the raw text of one section, produce routing metadata only.

Rules:
- summary describes WHAT THE READER WILL FIND in this section, in one
  sentence. It is metadata, not evidence. Do not paraphrase numbers.
- has_table is true only if the raw section contains a tabular layout
  (rows + columns of figures), not just a list.
- suggested_node_id is a short snake_case slug derived from the heading.
- translation in Korean.

Heading: {heading}
Page range: {start}-{end}
Raw section text:
---
{raw}
---"""

def architect_node(heading, raw_text, page_range, parent_path):
    """섹션 1개 → TreeNode 1개 (Architect LLM 호출 1회)"""
    prompt = _ARCHITECT_PROMPT.format(
        heading=heading,
        start=page_range[0],
        end=page_range[1],
        raw=raw_text[:8000],
    )
    resp = client.chat.completions.parse(
        model=MODEL_ARCHITECT,
        messages=[{"role":"user","content":prompt}],
        response_format=ArchitectDecision,    # ← schema 강제
        temperature=0,
    )
    decision = resp.choices[0].message.parsed
    node_id = ".".join(parent_path + [decision.suggested_node_id])
    return TreeNode(
        node_id=node_id,
        title=decision.title,
        summary=decision.summary,
        page_range=tuple(page_range),
        parent_path=parent_path,
        has_table=decision.has_table,
        source_refs=[f"{page_range[0]}-{page_range[1]}"],
    )

# 모든 섹션 처리 (실전은 parallel 호출, 데모는 순차)
tree = {}
print("\n--- Architect 실행 ---")
for s in SECTIONS:
    node = architect_node(s["heading"], s["raw_text"],
                          tuple(s["page_range"]), s["parent_path"])
    tree[node.node_id] = node
    (RAW_DIR / f"{node.node_id}.txt").write_text(s["raw_text"])
    print(f"  [{node.node_id}] L{len(node.parent_path)} {node.title}  "
          f"(table={node.has_table})")


--- Architect 실행 ---
  [management_discussion_and_analysis] L0 Item 7. Management Discussion and Analysis  (table=False)
  [mdna.results_of_operations] L1 Results of Operations  (table=False)
  [mdna.results_of_operations.quarterly_operating_income] L2 Quarterly Operating Income  (table=True)
  [mdna.results_of_operations.restructuring_charges] L2 Restructuring Charges  (table=False)
  [risk_factors] L0 Item 1A. Risk Factors  (table=False)


### 5-2-3. 부모-자식 관계 연결 + 트리 저장 + raw/summary 분리 검증

In [30]:
# ─────────────────────────────────────────────────────
# §5-2-3: 부모-자식 관계 + 트리 저장 + 분리 검증
# ─────────────────────────────────────────────────────
# 부모 노드의 children 리스트 채우기
for node in tree.values():
    if node.parent_path:
        parent_id = ".".join(node.parent_path)
        if parent_id in tree:
            tree[parent_id].children.append(node.node_id)

# 트리 JSON 저장
tree_json = RES / "tree.json"
with open(tree_json, "w") as f:
    json.dump({nid: n.model_dump() for nid, n in tree.items()}, f, indent=2)
print(f"트리 저장: {tree_json}")
print(f"raw 파일: {len(list(RAW_DIR.glob('*.txt')))}개 in {RAW_DIR}")

# raw vs summary 분리 검증 — 의도적으로 명시적 함수로 캡슐화
def load_raw_section(node_id) -> Optional[str]:
    """답변 시 source of truth — summary가 아닌 raw 참조"""
    p = RAW_DIR / f"{node_id}.txt"
    return p.read_text() if p.exists() else None

print("\n--- raw vs summary 분리 검증 ---")
sample = list(tree.values())[2]   # 3번째 노드 (Quarterly Operating Income)
print(f"node:    {sample.node_id}")
print(f"summary: {sample.summary[:80]} ... (← 라우팅 메타)")
print(f"raw:     {load_raw_section(sample.node_id)[:80]} ... (← 정답 소스)")

트리 저장: work/results/tree.json
raw 파일: 7개 in work/raw

--- raw vs summary 분리 검증 ---
node:    mdna.results_of_operations.quarterly_operating_income
summary: 이 분기별 영업이익과 구조조정 비용 제외 영업이익의 비교 표와 전년 동기 대비 변화 설명을 찾을 수 있습니다. ... (← 라우팅 메타)
raw:     Quarter | OpInc | OpInc ex-restructuring
Q3 2023 | $13,453M | $14,201M
Q3 2024 | ... (← 정답 소스)


---
## §5-3 Stage 2 — Reasoner (LangGraph + budget + trace)

LangGraph의 `StateGraph`로 추론 루프를 구성한다. 각 turn은 `traversal.jsonl`에 기록되어 완전한 관찰 가능성을 제공한다.

**Chapter 4와의 차이**:
- Chapter 4: 단순 4-노드 (analyze→descend→retrieve→generate)
- Chapter 5: budget·evidence_gap·trace 추가 + 3중 종료 조건

### 5-3-1. TraversalState — Annotated reducer 활용

`Annotated[..., operator.add]`로 누적 키와 덮어쓰기 키를 명시적으로 구분한다.

In [31]:
# ─────────────────────────────────────────────────────
# §5-3-1: TraversalState 정의 + 보조 함수
# ─────────────────────────────────────────────────────
DEFAULT_BUDGET = 8

def _merge_unique(existing, new):
    """LangGraph reducer — 중복 없이 누적"""
    seen = set(existing)
    return existing + [x for x in new if not (x in seen or seen.add(x))]

class TraversalState(TypedDict):
    """Stage 2의 공유 상태"""
    query:        str
    tree:         dict           # node_id → TreeNode
    frontier:     list[str]
    selected:     Annotated[list[str], _merge_unique]   # 누적
    evidence_gap: str            # 덮어씀
    budget_left:  int            # 단조 감소
    done:         bool

print("TraversalState 정의 완료")

TraversalState 정의 완료


### 5-3-2. 프롬프트 + CandidateView 추출 + Trace 함수

In [32]:
# ─────────────────────────────────────────────────────
# §5-3-2: Reasoner 프롬프트 + 헬퍼 + Trace
# ─────────────────────────────────────────────────────
_REASONER_PROMPT = """You are exploring a structured document to answer a query.
You see candidate nodes (title, summary, page range, parent path) — NOT raw text.
Your job is to (a) select nodes that look likely to contain the literal evidence
and (b) decide which child branches deserve to be expanded next.

Query: {query}
Current evidence gap: {gap}
Budget remaining (steps): {budget}
Candidates this step:
{candidates}

Rules:
- Prefer leaf nodes for select_for_evidence. Expand internal nodes via expand_next.
- has_enough_evidence is true only if the selected nodes plausibly contain
  every fact the query asks for, including comparison periods or exclusions.
- evidence_gap describes what is still missing in plain language.
- Translation in Korean."""

def _candidate_views(state):
    """frontier의 node_id들을 CandidateView 객체로 변환 (raw text 제외)"""
    return [
        CandidateView(**state["tree"][nid].model_dump())
        for nid in state["frontier"]
        if nid in state["tree"]
    ]

def _trace(turn, state, decision):
    """매 turn을 traversal.jsonl에 append"""
    rec = {
        "ts": time.time(), "turn": turn,
        "query": state["query"],
        "frontier": state["frontier"],
        "candidates": [c.node_id for c in _candidate_views(state)],
        "selected": decision.select_for_evidence,
        "expand_next": decision.expand_next,
        "reason": decision.reason,
        "evidence_gap": decision.evidence_gap,
        "budget_left": state["budget_left"],
    }
    with TRACE_PATH.open("a") as f:
        f.write(json.dumps(rec) + "\n")

### 5-3-3. reason_step 노드 + 라우팅 + 그래프 조립

종료 조건 3중: done=True, budget_left<=0, frontier=∅.

In [33]:
# ─────────────────────────────────────────────────────
# §5-3-3: LangGraph 그래프 조립
# ─────────────────────────────────────────────────────
from langgraph.graph import START, END, StateGraph

def reason_step(state: TraversalState) -> dict:
    """한 turn에 LLM 1회 호출 → TraversalDecision 반환"""
    candidates = _candidate_views(state)
    if not candidates:
        return {"done": True, "evidence_gap": "frontier_empty"}

    prompt = _REASONER_PROMPT.format(
        query=state["query"],
        gap=state["evidence_gap"] or "(none yet)",
        budget=state["budget_left"],
        candidates="\n".join(c.model_dump_json() for c in candidates),
    )
    resp = client.beta.chat.completions.parse(
        model=MODEL_REASONER,
        messages=[{"role":"user","content":prompt}],
        response_format=TraversalDecision,    # ← schema 강제
        temperature=0,
    )
    decision = resp.choices[0].message.parsed
    turn = DEFAULT_BUDGET - state["budget_left"] + 1
    _trace(turn, state, decision)

    # 다음 frontier 계산 — expand_next 노드들의 children
    next_frontier = []
    for nid in decision.expand_next:
        node = state["tree"].get(nid)
        if node is not None:
            next_frontier.extend(node.children)

    print(f"  turn {turn}: select={decision.select_for_evidence} "
          f"expand={decision.expand_next}  gap={decision.evidence_gap[:50]}")

    return {
        "selected":     decision.select_for_evidence,
        "frontier":     next_frontier,
        "evidence_gap": decision.evidence_gap,
        "budget_left":  state["budget_left"] - 1,
        "done":         decision.has_enough_evidence,
    }


def route_after_reason(state) -> Literal["reason", "end"]:
    """3중 종료 조건"""
    if state["done"]:           return "end"
    if state["budget_left"]<=0: return "end"
    if not state["frontier"]:   return "end"
    return "reason"

# 그래프 조립
graph = StateGraph(TraversalState)
graph.add_node("reason", reason_step)
graph.add_edge(START, "reason")
graph.add_conditional_edges("reason", route_after_reason,
    {"reason": "reason", "end": END})
traverse = graph.compile()
print("LangGraph 컴파일 완료")

LangGraph 컴파일 완료


### 5-3-4. 실제 쿼리 실행 + Trace 기록

In [34]:
# ─────────────────────────────────────────────────────
# §5-3-4: 실제 쿼리 실행
# ─────────────────────────────────────────────────────
TRACE_PATH.write_text("")    # 이전 trace 초기화

QUERY = ("What was Q3 2024 operating income excluding restructuring charges, "
         "and how did it change versus Q3 2023?")

# root 자식들로 frontier 초기화 (parent_path 비어있는 노드들)
root_children = [n.node_id for n in tree.values() if not n.parent_path]
initial = {
    "query": QUERY,
    "tree": tree,
    "frontier": root_children,
    "selected": [],
    "evidence_gap": "",
    "budget_left": DEFAULT_BUDGET,
    "done": False,
}

print(f"Q: {QUERY}\n")
print(f"frontier 시작: {root_children}\n")
print("--- 추론 루프 ---")
result = traverse.invoke(initial)
print(f"\n선택된 노드: {result['selected']}")
print(f"budget 사용: {DEFAULT_BUDGET - result['budget_left']}/{DEFAULT_BUDGET}")
print(f"done: {result['done']}")
print(f"\nTrace 파일: {TRACE_PATH}")
print(f"기록된 turn 수: {sum(1 for _ in TRACE_PATH.open())}")

Q: What was Q3 2024 operating income excluding restructuring charges, and how did it change versus Q3 2023?

frontier 시작: ['management_discussion_and_analysis', 'risk_factors']

--- 추론 루프 ---
  turn 1: select=['management_discussion_and_analysis'] expand=['management_discussion_and_analysis']  gap=Q3 2024의 'restructuring charges 제외' 영업이익 수치와 Q3 20

선택된 노드: ['management_discussion_and_analysis']
budget 사용: 1/8
done: False

Trace 파일: work/traversal.jsonl
기록된 turn 수: 1


---
## §5-4 Stage 3 — Answerer + Verifier (fail closed)

Answerer가 `GroundedAnswer`를 생성하면, Verifier가 5가지 결정적 검사를 수행한다. 하나라도 실패하면 즉시 refusal로 전환한다.

**왜 결정적 Verifier인가?**
- LLM-as-Judge보다 빠르고 저렴 (2차 LLM 호출 없음)
- 동일 입력에 동일 결과 (재현 가능)
- 해석 가능 (어느 검사가 실패했는지 명확)

### 5-4-1. Verifier — 5종 결정적 검사

이것이 본 챕터의 핵심 코드다. 5가지 모두 정규식·집합 연산·substring 검사만 사용 — LLM은 호출하지 않는다.

In [35]:
# ─────────────────────────────────────────────────────
# §5-4-1: Verifier 5종 결정적 검사
# ─────────────────────────────────────────────────────
_NUMBER = re.compile(r"-?\$?\d[\d,]*(?:\.\d+)?%?")
_EXCLUSION_HINTS = ("excluding", "ex-", "before", "without", "net of")

def verify(answer: GroundedAnswer, query: str, selected, tree, raw_by_id) -> VerifyResult:
    """5종 결정적 검사 — LLM 미사용"""
    failures = []
    selected_set = set(selected)

    for c in answer.citations:
        # (1) selected 외 인용 차단
        if c.node_id not in selected_set:
            failures.append(f"citation_node_not_selected:{c.node_id}")
            continue
        node = tree.get(c.node_id)
        if node is None:
            failures.append(f"citation_node_missing:{c.node_id}")
            continue
        # (2) page range 검증
        lo, hi = node.page_range
        if not (lo <= c.page <= hi):
            failures.append(
                f"citation_page_outside_range:{c.node_id}:{c.page} not in {lo}-{hi}"
            )
        # (3) substring 검증
        raw = raw_by_id.get(c.node_id, "")
        if c.quote.strip() and c.quote.strip() not in raw:
            failures.append(f"citation_quote_not_in_source:{c.node_id}")

    # (4) numeric grounding — 답변의 숫자가 인용문 숫자의 부분집합인가
    answer_numbers = set(_NUMBER.findall(answer.answer))
    cited_numbers = set()
    for c in answer.citations:
        cited_numbers.update(_NUMBER.findall(c.quote))
    ungrounded = answer_numbers - cited_numbers
    if ungrounded:
        failures.append(f"ungrounded_numeric_claims:{sorted(ungrounded)[:3]}")

    # (5) exclusion clause — query에 힌트가 있으면 답변에도 등장해야
    if any(h in query.lower() for h in _EXCLUSION_HINTS):
        text = (answer.answer + answer.calculation).lower()
        if not any(h in text for h in _EXCLUSION_HINTS):
            failures.append("exclusion_clause_not_addressed")

    return VerifyResult(ok=not failures, failures=failures)

print("Verifier 정의 완료 — 5종 결정적 검사")

Verifier 정의 완료 — 5종 결정적 검사


### 5-4-2. Answerer + Refusal — fail closed 패턴

Verifier 결과가 ok=False이면 즉시 refusal로 전환. confident hallucination이 사용자에게 도달하지 못하게 한다.

In [36]:
# ─────────────────────────────────────────────────────
# §5-4-2: Answerer + Refusal
# ─────────────────────────────────────────────────────
_ANSWER_PROMPT = """Answer the query using ONLY the source sections below.
Every numeric claim must be backed by a citation whose `quote` is a literal
substring of the cited section. If the sources do not contain the answer
(including any requested exclusions or comparison periods), set confidence to
"low" and list the missing facts in `missing_evidence`. Do not infer values
that are not in the source text and translation in Korean.

Query: {query}

Source sections:
{sources}"""

REFUSAL_TEMPLATE = (
    "Not enough evidence in the retrieved sections to answer with confidence. "
    "Verifier failures: {failures}. Missing facts: {missing}."
)

def _format_sources(selected, tree):
    """선택된 노드들의 raw text를 컨텍스트로 포맷"""
    blocks = []
    raw_by_id = {}
    for nid in selected:
        node = tree.get(nid)
        raw = load_raw_section(nid) if node else None
        if not node or raw is None:
            continue
        raw_by_id[nid] = raw
        header = f"[{nid}] {node.title} (pages {node.page_range[0]}-{node.page_range[1]})"
        blocks.append(f"{header}\n{raw}")
    return "\n\n".join(blocks), raw_by_id


def answer_from_evidence(query, selected, tree) -> GroundedAnswer:
    """Answerer + Verifier + Refusal의 통합 함수"""
    if not selected:
        # 선택된 노드 없음 → 즉시 refusal
        return GroundedAnswer(
            answer=REFUSAL_TEMPLATE.format(
                failures=["no_selected_nodes"], missing=["all"]),
            citations=[], confidence="low",
            missing_evidence=["traversal_returned_empty"],
        )

    sources_text, raw_by_id = _format_sources(selected, tree)
    resp = client.beta.chat.completions.parse(
        model=MODEL_ANSWERER,
        messages=[{"role":"user","content":_ANSWER_PROMPT.format(
            query=query, sources=sources_text)}],
        response_format=GroundedAnswer,        # ← schema 강제
        temperature=0,
    )
    candidate = resp.choices[0].message.parsed

    # ★ Verifier 호출
    result = verify(candidate, query, selected, tree, raw_by_id)
    if result.ok:
        return candidate                            # ✓ pass

    # ✗ fail closed → refusal
    return GroundedAnswer(
        answer=REFUSAL_TEMPLATE.format(
            failures=result.failures,
            missing=candidate.missing_evidence or ["unknown"]),
        calculation="",
        citations=candidate.citations,
        confidence="low",
        missing_evidence=candidate.missing_evidence + result.failures,
    )

print("Answerer + Verifier 정의 완료")

Answerer + Verifier 정의 완료


### 5-4-3. 실제 답변 생성 — Stage 1·2·3 통합 실행

In [37]:
# ─────────────────────────────────────────────────────
# §5-4-3: 실제 답변 생성 (3-Stage 통합)
# ─────────────────────────────────────────────────────
final = answer_from_evidence(QUERY, result["selected"], tree)

print("="*60)
print("  FINAL ANSWER (Three-Stage 결과)")
print("="*60)
print(f"confidence: {final.confidence}")
print(f"\n[answer]\n{final.answer}")
print(f"\n[calculation]\n{final.calculation}")
print(f"\n[citations]")
for c in final.citations:
    print(f"  • {c.node_id}, p.{c.page}: \"{c.quote[:80]}\"")
if final.missing_evidence:
    print(f"\n[missing_evidence]\n{final.missing_evidence}")

  FINAL ANSWER (Three-Stage 결과)
confidence: low

[answer]
Not enough evidence in the retrieved sections to answer with confidence. Verifier failures: ["ungrounded_numeric_claims:['2023', '2024', '3']"]. Missing facts: ['Q3 2024 operating income excluding restructuring charges', 'Q3 2023 operating income excluding restructuring charges', 'The year-over-year change between Q3 2024 and Q3 2023'].

[calculation]


[citations]
  • management_discussion_and_analysis, p.25: "See Results of Operations for quarterly detail."

[missing_evidence]
['Q3 2024 operating income excluding restructuring charges', 'Q3 2023 operating income excluding restructuring charges', 'The year-over-year change between Q3 2024 and Q3 2023', "ungrounded_numeric_claims:['2023', '2024', '3']"]


### 5-4-4. Verifier 동작 시연 — 의도적 hallucination 차단

일부러 부정확한 답변을 만들어 Verifier가 어떻게 잡아내는지 확인한다. 5가지 검사가 각각 어떤 케이스를 막는지 명확해진다.

In [38]:
# ─────────────────────────────────────────────────────
# §5-4-4: Verifier 동작 시연 — 의도적 hallucination 차단
# ─────────────────────────────────────────────────────

# 일부러 hallucination 답변을 만들어 Verifier가 잡는지 확인
malicious = GroundedAnswer(
    answer="Q3 2024 operating income excluding restructuring was $99,999M, up 50% YoY.",
    calculation="",
    citations=[Citation(
        node_id="mdna.results_of_operations.quarterly_operating_income",
        page=41,
        quote="completely fabricated quote that is not in source",  # ← raw에 없음
    )],
    confidence="high",
    missing_evidence=[],
)

raw_by_id = {nid: load_raw_section(nid) for nid in result["selected"]}
vr = verify(malicious, QUERY, result["selected"], tree, raw_by_id)
print("--- Verifier 결과 (malicious 답변 입력) ---")
print(f"ok: {vr.ok}")
print(f"failures: {vr.failures}")
print()
print("→ ok=False이므로 답변은 refusal로 전환됨 (confident hallucination 차단)")
print("→ 5가지 검사 중 어느 것이 잡았는지 failures 리스트로 추적 가능")

--- Verifier 결과 (malicious 답변 입력) ---
ok: False
failures: ['citation_node_not_selected:mdna.results_of_operations.quarterly_operating_income', "ungrounded_numeric_claims:['$99,999', '2024', '3']"]

→ ok=False이므로 답변은 refusal로 전환됨 (confident hallucination 차단)
→ 5가지 검사 중 어느 것이 잡았는지 failures 리스트로 추적 가능


### 5-4-5. Trace 파일 분석 — 모든 의사결정 추적

Stage 2의 traversal.jsonl을 읽어 모든 turn의 의사결정을 분석한다. 임베딩 RAG는 '왜 이 청크가 뽑혔는가'를 설명하지 못하지만, 본 시스템은 모든 결정이 자연어로 기록된다.

In [39]:
# ─────────────────────────────────────────────────────
# §5-4-5: Trace 파일 분석
# ─────────────────────────────────────────────────────
print("--- Trace 분석 ---")
turns = [json.loads(line) for line in TRACE_PATH.open()]
print(f"총 {len(turns)} turn 기록")
for t in turns:
    print(f"  turn {t['turn']}: candidates={t['candidates']} → selected={t['selected']}")
print()
print(f"\n저장 경로: {TRACE_PATH}")
print(f"이 파일이 디버깅의 핵심 — 모든 의사결정이 자연어로 기록")

--- Trace 분석 ---
총 1 turn 기록
  turn 1: candidates=['management_discussion_and_analysis', 'risk_factors'] → selected=['management_discussion_and_analysis']


저장 경로: work/traversal.jsonl
이 파일이 디버깅의 핵심 — 모든 의사결정이 자연어로 기록
